# GLiNER2 한국어 추출: LLM 증류

[한국어 미세조정](../notebooks/02_korean_fine_tuning.ipynb)에서는 사람이 단 KLUE-NER 정답으로 LoRA 어댑터를 학습해 KLUE 검증 데이터의 F1을 0.580에서 0.812로 올렸다. 사람이 단 정답은 구하기 어렵다. 그래서 LLM이 정답을 대신 달고 그 정답으로 GLiNER를 학습하는 **증류**(distillation, LLM 라벨 학습)가 현실적인 대안으로 자주 쓰인다.

본 실험은 사람 정답과 LLM 정답의 효과를 같은 조건에서 비교한다. **같은 KLUE 학습 2,000문장**에 사람 정답 대신 `gpt-5.4-mini`가 단 정답으로 어댑터를 학습하고, KLUE 검증 5,000문장의 사람 정답으로 평가한다. 문장과 학습 설정이 같으므로, 점수 차이는 정답(라벨) 출처의 차이로 해석할 수 있다.

| 역할 | 내용 |
|---|---|
| 선생님 | `gpt-5.4-mini`. 사람·기관·장소를 표시한다 |
| 학생 | `fastino/gliner2.5-multi-v1` + Kiwi 분할기 + LoRA |
| 시험지 | KLUE-NER 검증 5,000문장, 사람 정답 8,249개 |

## 결과 요약

> 아래 수치는 2026-11-11 실행 결과(`llm_distillation.json`)이다. 학습 2,000문장 × 2 epoch, 평가는 KLUE 검증 5,000문장 전체, Darwin arm64 CPU.

| 조건 | 학습 정답 | person F1 | organization F1 | location F1 | 전체 P / R / F1 |
|---|---|---|---|---|---|
| 기본 | 없음 | 0.561 | 0.599 | 0.599 | 0.543 / 0.623 / **0.580** |
| KLUE 어댑터 | 사람 | **0.857** | **0.759** | **0.763** | 0.818 / 0.806 / **0.812** |
| 증류 어댑터 | gpt-5.4-mini | 0.780 | 0.607 | 0.549 | 0.642 / 0.719 / **0.678** |
| (참고) gpt-5.4-mini 자체* | – | 0.789 | 0.603 | 0.567 | 0.634 / 0.724 / **0.676** |

\* 검증 데이터가 아니라 학습 2,000문장에서 선생님 정답을 사람 정답과 비교한 값이다. 비용을 줄이려고 검증 데이터에서는 LLM을 호출하지 않았다.

- **증류 어댑터는 선생님과 비슷한 수준에 이른다.** 학생의 검증 F1(0.678)이 선생님이 학습 문장에서 사람 정답과 일치한 정도(0.676)와 비슷하다. 두 값은 서로 다른 문장에서 잰 것이므로 대략적인 비교이다.
- **사람 정답으로 학습한 어댑터보다 F1이 0.134 낮다.** 기본 모델보다는 0.098 높다. 학습 전과 사람 정답 학습 사이 격차의 약 42%를 메웠다.
- **선생님의 기준 차이가 학생에게 측정 가능하게 옮겨 간다.** 선생님이 사람 정답과 다르게 표시한 상위 사례는 대부분 표기 기준의 차이이다(`박씨`·`A씨`의 호칭 포함, 주소 분할, 국가의 타입, `경찰`·`정부` 누락). 증류 어댑터 오답의 약 28%가 선생님도 학습 문장에서 틀린 이름과 같은 이름에서 나왔다. 대조군인 KLUE 어댑터는 15~16%이다.
- **비용은 작다.** 2,000문장 라벨링에 1.29달러, 약 4분이 들었다. 이후 GLiNER 추론은 문장당 약 0.05초(CPU, 순차 처리)이고 API 비용이 없다. LLM은 호출당 지연이 약 1.8초이고, 16개 동시 호출 시 처리량은 문장당 약 0.11초였다.

## 실험 조건

| 항목 | 값 |
|---|---|
| 학습 문장 | KLUE-NER 학습 데이터에서 `random.seed(42)`로 뽑은 2,000문장. [한국어 미세조정](../notebooks/02_korean_fine_tuning.ipynb)과 같은 문장 |
| 선생님 | `gpt-5.4-mini`, Responses API, 구조화 출력, 추론 강도 기본값, 동시 호출 16 |
| 선생님 지시문 | KLUE 표기 규칙(타입 정의, 범위 규칙, 문맥에 따른 타입 결정)을 글로 자세히 설명한다. **예시 문장(few-shot)과 KLUE 문장은 넣지 않는다.** 지시문 전문은 [`config/teacher.py`](../config/teacher.py) |
| 위치 붙이기 | 선생님이 돌려준 원문 표면형을 문장에서 찾아 위치를 붙인다. 찾지 못하면 버린다 |
| 학생 학습 | KLUE 어댑터와 같은 설정. LoRA, 2 epoch, 배치 8, seed 42, Kiwi 경계와 맞지 않는 범위 제외 |
| 평가 | KLUE 검증 5,000문장, 범위와 타입이 정확히 일치해야 정답, threshold 0.5 |
| 단가 | 입력 0.75달러, 출력 4.50달러 (100만 토큰당, [OpenAI 가격표](https://developers.openai.com/api/docs/pricing), 2026-11-11 확인) |

실행은 [`scripts/run_llm_distillation.py`](../scripts/run_llm_distillation.py)가 담당한다. 단계별 결과를 `outputs/distill/`에 저장하므로 중간에 멈춰도 API를 다시 호출하지 않고 이어서 실행할 수 있다. 이 노트북은 저장된 결과 파일을 불러와 분석만 한다.

## 준비

### 경고 출력 설정

라이브러리가 출력하는 안내성 경고를 숨긴다.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.distributed.elastic").setLevel(logging.ERROR)

### 프로젝트 경로와 결과 파일

현재 작업 디렉터리에서 상위로 올라가며 `pyproject.toml`이 있는 프로젝트 루트를 찾는다. `RUN_EXPERIMENT = True`로 바꾸면 실행 명령을 안내한다. 라벨링 2,000회 API 호출과 학습·평가를 포함하므로 노트북 안에서는 실행하지 않는다.

In [ ]:
import json
from pathlib import Path

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
RESULT_PATH = PROJECT / "experiments/results/llm_distillation.json"
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    print("프로젝트 루트에서 실행한다:")
    print("uv run python scripts/run_llm_distillation.py --price-input 0.75 --price-output 4.5 "
          "--price-source https://developers.openai.com/api/docs/pricing")

result = json.loads(RESULT_PATH.read_text())
run = result["run"]
print(f"✓ Result file: {RESULT_PATH.relative_to(PROJECT)}")
print(f"  {run['date']}  teacher={run['model']}  train={run['train_size']}  "
      f"eval={run['eval_size']} (entities {run['eval_entities']})  {run['device']}")

## 선생님 라벨링

선생님이 학습 2,000문장에 단 정답의 통계와, 같은 문장의 사람 정답과 비교한 일치도이다.

In [ ]:
labeling = result["labeling"]
teacher = result["teacher_vs_human_train"]

print(f"sentences {labeling['sentences']}  failed {labeling['failed']}  "
      f"dropped (surface not in text) {labeling['dropped']}")
print(f"entities {labeling['total_spans']}  Kiwi-aligned {labeling['aligned']} "
      f"({labeling['aligned'] / labeling['total_spans']:.1%})")
print(f"tokens in {labeling['input_tokens']:,} / out {labeling['output_tokens']:,}  "
      f"cost ${labeling['cost_usd']:.2f}  wall {labeling['wall_seconds']:.0f}s  "
      f"mean call {labeling['mean_call_seconds']:.2f}s")
print()
print("teacher vs human labels (train 2,000 sentences)")
for label in ["person", "organization", "location", "overall"]:
    s = teacher[label]
    print(f"  {label:>12}: P {s['precision']:.3f}  R {s['recall']:.3f}  F1 {s['f1']:.3f}")

| 항목 | 값 |
|---|---|
| 실패 호출 | 0 / 2,000 |
| 표시한 엔티티 | 3,182개 (원문에서 찾지 못해 버린 것 7개) |
| Kiwi 경계와 맞는 비율 | 98.5% (3,134개) |
| 선생님 대 사람 정답 F1 | person 0.789, organization 0.603, location 0.567, 전체 **0.676** |

선생님은 조사를 빼고 원문 표면형을 유지하라는 규칙은 잘 지켰다. 원문에서 찾지 못한 표면형은 7개뿐이다. 그러나 사람 정답과의 일치도는 0.676에 그친다. 아래는 선생님이 사람 정답과 다르게 표시한 이름의 상위 사례이다(결과 파일의 `teacher_disagreements`).

| 차이 유형 | 선생님만 표시 (빈도) | 사람만 표시 (빈도) |
|---|---|---|
| 호칭·성만 표시하는 규칙 | `A씨` 18, `김씨` 14, `이씨` 9, `박씨` 6 | `A` 44, `이` 27, `김` 27, `B` 14, `박` 11 |
| 주소를 나누어 표시 | `서울` 21, `미국` 18, `경기도` 7, `뉴욕` 6 (KLUE는 `서울 강남구`처럼 한 범위) | – |
| 국가의 타입 | `북한` 14, `러시아` 9, `중국` 5 (장소) | `북한` 12, `러시아` 9 (기관) |
| 일반 기관명·국가 생략 | – | `경찰` 22, `정부` 8, `한국` 25, `일본` 16 |
| 사건·서비스·작품 이름 | `세월호` 11, `인스타그램` 7 | – |

선생님만 표시한 예측은 1,166개, 사람만 표시한 정답은 767개이다. 상위 사례의 대부분은 오답이라기보다 기준의 차이이다. 지시문에 "호칭은 범위에서 뺀다"고 적었지만 선생님은 `A씨`·`김씨`를 하나의 이름으로 보았다. 또 "고유한 이름이 아닌 일반 명사구는 표시하지 않는다"는 규칙 때문에, KLUE가 기관으로 표시하는 `경찰`·`정부`를 생략했다. `한국`·`일본`을 생략한 경우도 많아서, 선생님이 국가를 일관되게 장소로 본 것도 아니다. **글로 된 규칙만으로는 데이터셋 고유의 경계 사례를 전달하기 어렵다.**

## 학생 학습

선생님 정답으로 LoRA 어댑터를 학습한다. 설정은 KLUE 어댑터와 같다.

In [ ]:
training = result["training"]
print(f"examples {training['examples']}  steps {training['total_steps']}  "
      f"time {training['seconds']:.0f}s ({training['samples_per_second']:.1f} samples/s)  "
      f"trainable params {training['trainable_params']:,}")

2,000문장 × 2 epoch(500 단계)에 CPU로 221초가 걸렸다. 학습 파라미터는 2,715,664개로, [한국어 미세조정](../notebooks/02_korean_fine_tuning.ipynb)의 KLUE 어댑터(전체의 0.94%)와 같은 LoRA 설정이다.

## 조건별 성능

네 조건을 같은 기준으로 비교한다. gpt-5.4-mini 행은 검증 데이터가 아니라 학습 문장에서 잰 값이다.

In [ ]:
LABELS = run["labels"]
rows = {**result["conditions"], "teacher (train)*": result["teacher_vs_human_train"]}

print(f"{'condition':<18}" + "".join(f"{label:>14}" for label in LABELS)
      + f"{'P':>8}{'R':>8}{'F1':>8}{'particles':>11}")
for name, s in rows.items():
    cells = "".join(f"{s[label]['f1']:>14.3f}" for label in LABELS)
    o = s["overall"]
    print(f"{name:<18}{cells}{o['precision']:>8.3f}{o['recall']:>8.3f}{o['f1']:>8.3f}{s['particles']:>11}")

| 조건 | person | organization | location | 정밀도 | 재현율 | 전체 F1 | 조사 포함 예측 |
|---|---|---|---|---|---|---|---|
| 기본 | 0.561 | 0.599 | 0.599 | 0.543 | 0.623 | 0.580 | 84 |
| KLUE 어댑터 | 0.857 | 0.759 | 0.763 | 0.818 | 0.806 | **0.812** | 105 |
| 증류 어댑터 | 0.780 | 0.607 | 0.549 | 0.642 | 0.719 | **0.678** | 99 |
| gpt-5.4-mini (학습 문장, 2,000문장)* | 0.789 | 0.603 | 0.567 | 0.634 | 0.724 | 0.676 | 22 |

- KLUE 어댑터는 이전 측정값(0.812)을 그대로 재현했다. 학습 문장과 설정이 같아서, 증류 어댑터와의 차이는 정답 출처의 차이로 볼 수 있다.
- **증류 어댑터의 레이블별 점수는 선생님의 패턴을 거의 그대로 따른다.** person이 가장 높고 location이 가장 낮다. 주소 분할과 국가 타입의 차이가 location과 organization 점수를 함께 낮춘다.
- 조사 포함 예측 수는 평가 문장 수가 다르므로(검증 5,000 대 학습 2,000) 선생님 행과 직접 비교하지 않는다.
- person은 기본 모델 대비 크게 올랐다(0.561 → 0.780). 다만 호칭·성만 표시하는 규칙의 차이 때문에 KLUE 어댑터(0.857)와는 차이가 남는다.
- organization(0.599 → 0.607)과 location(0.599 → 0.549)은 거의 오르지 않거나 오히려 떨어졌다. 기준 차이가 큰 타입에서는 증류가 KLUE 점수를 올리지 못한다.

## 오류 분류

정답이 아닌 예측과 놓친 정답을 유형별로 센다. 하나의 오답은 한 유형으로만 센다. 우선순위는 타입 오류 → 범위가 넓음 → 범위가 좁음 → 엉뚱한 예측이고, 어느 예측과도 짝지어지지 않은 정답은 놓침으로 센다.

In [ ]:
KINDS = ["type", "wider", "narrower", "spurious", "missed"]
print(f"{'condition':<16}" + "".join(f"{kind:>10}" for kind in KINDS))
for name, counts in result["errors"].items():
    print(f"{name:<16}" + "".join(f"{counts[kind]:>10}" for kind in KINDS))
print()
print("errors on surfaces the teacher also got wrong on train (KLUE adapter = control)")
for name, share in result["inherited_errors"].items():
    print(f"  {name:<14} extra predictions {share['fp']:>5} ({share['fp_inherited']:.1%} inherited)  "
          f"missed gold {share['fn']:>5} ({share['fn_inherited']:.1%} inherited)")

| 조건 (평가 범위) | 타입 오류 | 범위가 넓음 | 범위가 좁음 | 엉뚱한 예측 | 놓침 |
|---|---|---|---|---|---|
| KLUE 어댑터 (검증 5,000문장) | 245 | 302 | 232 | 695 | 824 |
| 증류 어댑터 (검증 5,000문장) | 364 | 566 | 551 | **1,820** | 839 |
| 선생님 (학습 2,000문장) | 82 | 167 | 135 | 782 | 383 |

- 증류 어댑터의 오답은 **엉뚱한 예측**이 가장 많다(KLUE 어댑터의 약 2.6배). 선생님도 같은 유형이 가장 많다.
- 범위 오류(넓음 566, 좁음 551)도 KLUE 어댑터의 약 두 배이다. `박씨`·`A 씨`의 호칭, `LPGA 투어`처럼 공식 이름의 범위를 어디까지 보는지가 다르다.
- 정밀도뿐 아니라 재현율도 떨어졌다(0.806 → 0.719). 놓침 수는 비슷하지만(824 대 839), 범위와 타입이 틀린 예측은 그 정답을 놓친 것으로도 계산되기 때문이다.

**선생님에게서 물려받은 오답**

증류 어댑터의 오답 중, 선생님이 학습 문장에서 틀린 것과 같은 (타입, 이름)에서 나온 비율을 센다. 흔한 이름은 우연히 겹칠 수 있으므로, 같은 방식으로 센 KLUE 어댑터의 비율을 대조군으로 둔다.

| 조건 | 정답에 없는 예측 | 그중 선생님과 같은 오답 | 놓친 정답 | 그중 선생님과 같은 오답 |
|---|---|---|---|---|
| KLUE 어댑터 (대조군) | 1,474 | 16.5% | 1,603 | 14.8% |
| 증류 어댑터 | 3,301 | **27.6%** | 2,320 | **28.3%** |

증류 어댑터는 대조군보다 약 11~13%p 더 많이 선생님과 같은 이름에서 틀린다. 선생님의 기준 차이가 학생에게 옮겨 간 것을 직접 보여주는 수치이다. 다만 나머지 약 72%는 선생님의 오답과 이름이 겹치지 않는다. 학생은 개별 이름을 외운 것이 아니라 선생님의 표시 경향(호칭 포함, 주소 분할 등)을 처음 보는 이름에까지 일반화했다고 볼 수 있다. 이 부분은 이름 단위 비교로는 잡히지 않으므로, 이 실험으로는 "오답의 몇 %가 기준 차이인가"를 정확히 나누지 못한다.

## 증류 전후 비교 예시

KLUE 어댑터와 증류 어댑터의 결과가 다른 검증 문장에서, 증류 어댑터가 정답과 일치한 문장과 KLUE 어댑터만 정답과 일치한 문장을 골랐다.

In [ ]:
NAMES = [("gold", "정답"), ("base", "기본"), ("klue_lora", "KLUE"), ("distill_lora", "증류")]
for example in result["examples"]:
    print(example["text"])
    for key, name in NAMES:
        found = ", ".join(f"{text}({label[:3]})" for text, label in example[key]) or "–"
        print(f"  {name:>4}: {found}")
    print()

| 문장 (요약) | 정답 | 기본 | KLUE 어댑터 | 증류 어댑터 |
|---|---|---|---|---|
| 엠비씨에서 사극이 이렇게 망한다는… | 엠비씨(org) | 엠비씨, `사극`(per) | 놓침 | **정답과 일치** |
| 춘천시와 원주시는 … 태백시는 … | 세 도시(loc) | 정답과 일치 | 세 도시를 **기관**으로 | **정답과 일치** |
| 지난주 북한이 … 미사일 4발을 발사한… | 북한(loc) | 정답과 일치 | 북한을 **기관**으로 | **정답과 일치** |
| 스노우가 공개한 사진은 댓글만… | 스노우(org) | 정답과 일치 | 놓침 | **정답과 일치** |
| 박씨의 딸(32)과 … 김모(33)씨 등… | 경찰, 박, 김 | `박씨`, `김모` | **정답과 일치** | `박씨`, `김모` |
| B 씨는 … A 씨를 더 의심하지… | B, A | `B 씨`, `A 씨`, `대학`(org) | **정답과 일치** | `B 씨`, `A 씨` |
| … 장하나를 따돌리고 LPGA 투어 첫 우승의… | 최운정, 장하나, LPGA | `보기`(loc), `LPGA 투어` | **정답과 일치** | `LPGA 투어` |
| 전자제품 대기업에서 일하는 최모(32)씨는… | 최 | `전자제품 대기업`(org), `최모` | **정답과 일치** | `전자제품 대기업`, `최모` |

- **증류 어댑터가 나은 경우**: 지명을 장소로 판단(`춘천시`, `북한`)하고, KLUE 어댑터가 놓친 이름(`엠비씨`, `스노우`)을 찾았다. 선생님의 타입 판단이 학습된 것으로 보인다. KLUE 어댑터는 국가·도시를 문맥에 따라 기관으로 표시하는 KLUE 규칙을 장소 문맥에까지 적용한 것으로 보인다.
- **KLUE 어댑터가 나은 경우**: `박씨` → `박`, `A 씨` → `A`처럼 데이터셋 고유의 표기 규칙. 이 규칙은 지시문에 "호칭 제외"로 적혀 있었지만, 선생님은 `씨`를 이름의 일부로 보는 경우가 많았다. `전자제품 대기업`처럼 일반 명사구를 기관으로 표시하는 오류도 증류 쪽에 남았다.
- 증류 어댑터는 "틀렸다"기보다 **선생님의 기준을 충실히 배웠다**. 그 기준이 KLUE 기준과 다른 만큼 KLUE 점수가 낮게 나온다.

## 자원과 비용

증류 비용은 라벨링부터로 정의한다. 학습 문장은 기존 KLUE 원문을 쓰므로 수집 비용이 없다. 로컬 CPU의 학습·추론 시간은 금액으로 환산하지 않는다.

In [ ]:
conditions = result["conditions"]
per_sentence_llm = labeling["cost_per_sentence_usd"]
print(f"labeling: ${labeling['cost_usd']:.2f} for {labeling['sentences']} sentences "
      f"(${per_sentence_llm * 1000:.2f} per 1,000), wall {labeling['wall_seconds'] / 60:.1f} min")
print(f"training: {training['seconds'] / 60:.1f} min on CPU")
print(f"latency per call: LLM {labeling['mean_call_seconds']:.2f}s  |  "
      f"GLiNER LoRA {conditions['distill_lora']['seconds_per_sentence']:.3f}s")
print(f"throughput per sentence: LLM {labeling['throughput_seconds_per_sentence']:.3f}s (concurrent)  |  "
      f"GLiNER LoRA {conditions['distill_lora']['seconds_per_sentence']:.3f}s (sequential)")
print(f"cost per sentence: LLM ${per_sentence_llm:.5f}  |  GLiNER $0")
print(f"break-even: {result['breakeven_sentences']:,} inference sentences")

| 항목 | 값 |
|---|---|
| 라벨링 토큰 | 입력 1,377,134 / 출력 57,038 (문장당 약 689 / 29) |
| 라벨링 비용 | **1.29달러** (1,000문장당 0.64달러) |
| 라벨링 시간 | 약 3.8분 (동시 호출 16, 호출당 평균 1.78초) |
| 학생 학습 시간 | 약 3.7분 (CPU) |
| 추론 지연 (호출 1회) | gpt-5.4-mini 약 1.78초 / 증류 GLiNER 약 0.046초(CPU) |
| 추론 처리량 (문장당) | gpt-5.4-mini 약 0.113초 (16개 동시 호출) / 증류 GLiNER 약 0.046초 (순차 처리) |
| 추론 비용 (문장당) | gpt-5.4-mini 0.00064달러 / 증류 GLiNER 0원 |
| 손익분기 | 추론 **2,000문장** |

- 입력 토큰의 대부분은 매 호출에 반복되는 지시문(약 650토큰)이다. 지시문이 1,024토큰 이상이면 OpenAI가 자동으로 캐시해 입력 단가가 10분의 1로 내려가므로, 지시문을 더 자세히 써도 비용이 크게 늘지 않을 수 있다.
- 손익분기는 "라벨링 비용 ÷ LLM 추론 1문장 비용"이다. 라벨링한 문장 수와 같아지는데, LLM으로 라벨링하는 것과 LLM으로 추론하는 것이 같은 호출이기 때문이다. 즉 **학습에 쓴 만큼의 문장을 GLiNER로 처리하면 라벨링 비용을 회수한다.** 그 이후부터는 API 비용이 들지 않는다. 속도는 호출 1회의 지연 기준으로 약 39배 빠르고, LLM을 16개 동시에 호출하는 처리량과 비교하면 약 2.5배 빠르다.
- Batch API를 쓰면 단가가 절반이 되어 라벨링 비용은 약 0.65달러가 된다. 대신 결과를 받기까지 몇 시간이 걸릴 수 있다.

## 결론과 한계

**결론**

1. LLM 정답으로 학습한 GLiNER는 **선생님과 비슷한 수준**에 이른다. 증류 어댑터의 검증 F1은 0.678이고, 선생님이 학습 문장에서 사람 정답과 일치한 정도는 0.676이다(서로 다른 문장에서 잰 값). 라벨링 1.29달러(약 4분)와 CPU 학습 약 4분으로, 문장당 약 0.05초에 API 비용 없이 쓸 수 있는 추출기를 얻는다.
2. 사람 정답으로 학습한 어댑터(0.812)에는 미치지 못한다. 선생님이 사람 정답과 다르게 표시한 상위 사례는 대부분 데이터셋 고유의 표기 기준(호칭과 성만 표시, 주소 범위, 국가 타입, 일반 기관명)이며, 증류 어댑터는 대조군보다 약 11~13%p 더 많이 선생님과 같은 이름에서 틀렸다. 기준 차이가 격차의 주요 원인 중 하나로 보이지만, 그 비율을 정확히 분리하지는 못했다. 글로 된 규칙만으로는 이런 경계 사례를 충분히 전달하지 못했다.
3. 따라서 증류의 품질은 **선생님의 기준이 평가 기준과 얼마나 같은가**로 정해진다. 평가 기준이 우리 서비스의 기준이라면(예: 대화 기억에서 서비스명을 기관으로 저장하고 싶다면) 증류 어댑터의 "오답" 중 상당수가 오히려 원하는 결과일 수 있다.

**한계**

- 선생님 품질은 검증 데이터가 아니라 학습 문장에서 쟀다(비용 절감). 두 데이터는 같은 KLUE 분포이지만 같은 시험지는 아니다.
- 단일 seed, 학습 2,000문장, 한 가지 지시문만 실험했다. 지시문 개선(경계 사례 규칙 추가)이나 few-shot, 더 큰 선생님 모델의 효과는 측정하지 않았다.
- 평가는 KLUE 도메인(뉴스·영화 리뷰)과 사람·기관·장소에 한정된다. 대화 도메인이나 관계 추출에서의 증류 효과는 확인하지 않았다.
- 오류 분류와 물려받은 오답 비율은 범위·타입·이름을 기계적으로 비교한 것이다. 선생님의 경향을 새 이름에 일반화한 오답은 이름 단위 비교로 잡히지 않는다.
- 오류 분류는 같은 예측에 대해서도 실행마다 몇 건씩 달라질 수 있다(집합 순회 순서에 따른 짝짓기 차이).